# Methodology: Optimization Formulation for Renewable Energy Communities

## 4.1 Overview of the Two-Stage Optimization Approach

The economic viability and operational efficiency of a renewable energy community depend critically on how internal energy flows are coordinated and how costs are allocated among heterogeneous participants. To address this complexity, this thesis employs a **two-stage hierarchical optimization framework** that balances individual participant incentives with collective community benefits.

The framework reflects the regulatory and economic reality of Austrian RECs: participants are independently metered and contracted with external suppliers, yet they benefit from internal electricity sharing at reduced grid tariffs. The two-stage approach ensures that collective optimization does not disadvantage any individual member — a fairness principle aligned with **Pareto optimality** and EU directives on energy community governance.

The optimization objective is modelled as a **Mixed-Integer Linear Program (MILP)** embedded within a **Model Predictive Control (MPC)** framework $^{[1,2]}$. The solver employed is **Gurobi**, a commercial optimization engine suitable for large-scale MILP problems.

---

## 4.2 Problem Structure

An REC comprises two participant categories, each with distinct decision-making scope:

- **Consumers** (set $\mathcal{C}$): entities with electricity load $L_{i,t}$ only, relying entirely on external grid supply or internal community purchases to meet their demand.
- **Prosumers** (set $\mathcal{P}$): entities with electricity load $L_{i,t}$, decentralised PV generation $G_{i,t}$, and optionally behind-the-meter battery energy storage systems (BESS), enabling them to modulate their net demand through generation, storage dispatch, and internal community participation.

The case study comprises **nine participants**: three prosumers (with PV systems) and six consumers. This consumer-dominated composition is representative of early-stage Austrian RECs, where renewable generation is concentrated among a subset of members and the broader membership participates primarily through internal electricity sharing.

The optimization is formulated as a MILP solved over a **rolling horizon of 24 hours at 15-minute resolution**, consistent with Austrian smart meter granularity and wholesale market settlement intervals.

---

## 4.3 Stage 1: Individual Optimization (Baseline Self-Cost Minimization)

### 4.3.1 Problem Formulation

In the first stage, each REC participant optimizes its energy bill **independently**, without reference to community peers or collective strategies. This stage establishes a baseline cost for each member and ensures that subsequent collective optimization respects individual incentives.

**Given parameters:**

- Discrete time horizon $\mathcal{T} = \{0, \ldots, T-1\}$ with step size $\Delta t = 0.25$ hours (15 minutes)
- Load profile $L_{i,t}$ (kW) and PV generation profile $G_{i,t}$ (kW) for prosumer $i \in \mathcal{P}$
- Grid import tariff $p^{\mathrm{imp}}_{i,t}$ (€/kWh) and export price $p^{\mathrm{exp}}_{i,t}$ (€/kWh), which may reflect time-of-use (ToU) or dynamic pricing structures
- Battery specifications for prosumer $i$ (if equipped):
  - Usable energy capacity $E^{\mathrm{bat}}_{\max,i}$ (kWh)
  - Charge power limit $P^{\mathrm{ch}}_{\max,i}$ and discharge power limit $P^{\mathrm{dis}}_{\max,i}$ (kW)
  - Round-trip efficiency $\eta^{\mathrm{ch}}_i$ (charge) and $\eta^{\mathrm{dis}}_i$ (discharge), typically in the range 0.85–0.95
  - Initial state of charge $E_{i,0}$ (kWh)

**Decision variables per participant $i$:**

| Variable | Description | Unit |
|----------|-------------|------|
| $x^{\mathrm{ch}}_{i,t} \geq 0$ | Charge power delivered to battery | kW |
| $x^{\mathrm{dis}}_{i,t} \geq 0$ | Discharge power extracted from battery | kW |
| $P^{\mathrm{imp}}_{i,t} \geq 0$ | Net electricity imported from the grid | kW |
| $P^{\mathrm{exp}}_{i,t} \geq 0$ | Net electricity exported to the grid | kW |
| $E_{i,t} \geq 0$ | Battery state of charge at end of time step $t$ | kWh |

**Objective function (cost minimization):**

$$\min_{x^{\mathrm{ch}}, x^{\mathrm{dis}}, P^{\mathrm{imp}}, P^{\mathrm{exp}}, E} \sum_{t \in \mathcal{T}} \Delta t \left( p^{\mathrm{imp}}_{i,t}\, P^{\mathrm{imp}}_{i,t} - p^{\mathrm{exp}}_{i,t}\, P^{\mathrm{exp}}_{i,t} \right) \quad \forall i \in \mathcal{C} \cup \mathcal{P}$$

Each participant $i$ independently minimizes their net electricity cost over the horizon $\mathcal{T}$. The subscript of the $\min$ operator lists the **five families of decision variables** that the solver optimizes jointly:

- $x^{\mathrm{ch}}, x^{\mathrm{dis}}$: battery charge and discharge schedules (zero for consumers, who have no BESS)
- $P^{\mathrm{imp}}, P^{\mathrm{exp}}$: grid exchange quantities, shaped by battery dispatch and local generation
- $E$: battery state-of-charge trajectory, linking consecutive time steps via the SOC dynamics constraint

The scalar $\Delta t = 0.25$ h converts power (kW) to energy (kWh), so each summand is a monetary cost in €. The first term $p^{\mathrm{imp}}_{i,t} P^{\mathrm{imp}}_{i,t}$ is the **cost of grid purchases**; the second term $p^{\mathrm{exp}}_{i,t} P^{\mathrm{exp}}_{i,t}$ (subtracted) is the **revenue from grid sales**, e.g. feed-in of surplus PV. The optimizer can reduce the objective by discharging stored energy to offset import, or by exporting surplus generation — subject to the energy balance and battery constraints below.

For consumers ($i \in \mathcal{C}$), the battery variables are absent and the objective reduces to $\min \sum_t \Delta t\, p^{\mathrm{imp}}_{i,t} L_{i,t}$, which is a constant given fixed load; Stage 1 therefore provides consumers with their baseline cost without any active dispatch decision.


### 4.3.2 Constraints

**Energy balance — prosumers:** For each prosumer $i \in \mathcal{P}$, at each time step $t$:

$$P^{\mathrm{imp}}_{i,t} - P^{\mathrm{exp}}_{i,t} + x^{\mathrm{dis}}_{i,t} - x^{\mathrm{ch}}_{i,t} = L_{i,t} - G_{i,t} \quad \forall t \in \mathcal{T}$$

This is Kirchhoff's power balance at the prosumer's meter point. The left-hand side captures all controllable power flows: net grid exchange ($P^{\mathrm{imp}} - P^{\mathrm{exp}}$) and net battery exchange ($x^{\mathrm{dis}} - x^{\mathrm{ch}}$). The right-hand side is the **net demand** — load minus local PV generation. When $G_{i,t} > L_{i,t}$ (surplus PV), the right-hand side is negative, meaning the prosumer must absorb the surplus through charging, grid export, or a combination. When $G_{i,t} < L_{i,t}$ (deficit), the prosumer draws from the grid, discharges the battery, or both.

**Energy balance — consumers:** For each consumer $i \in \mathcal{C}$:

$$P^{\mathrm{imp}}_{i,t} = L_{i,t}, \qquad P^{\mathrm{exp}}_{i,t} = 0 \quad \forall t \in \mathcal{T}$$

Consumers have no local generation or battery storage. Their entire electricity demand $L_{i,t}$ must therefore be covered by grid import. This constraint reflects the passive role of consumers in Stage 1 — they can reduce costs only by shifting load or purchasing internally from prosumers (in Stage 2), not through local dispatch.

**Battery state-of-charge (SOC) dynamics:**

$$E_{i,t} = E_{i,t-1} + \eta^{\mathrm{ch}}_i\, x^{\mathrm{ch}}_{i,t}\, \Delta t - \frac{1}{\eta^{\mathrm{dis}}_i}\, x^{\mathrm{dis}}_{i,t}\, \Delta t \quad \forall t \in \mathcal{T},\ i \in \mathcal{P}$$

This recurrence links consecutive SOC values, making the battery a **dynamic state variable** that couples decisions across time steps. Energy stored during charging is discounted by $\eta^{\mathrm{ch}}_i < 1$ (losses on the way in); energy extracted during discharging requires drawing $x^{\mathrm{dis}}_{i,t} / \eta^{\mathrm{dis}}_i$ from the stored reserve (losses on the way out). The product $\eta^{\mathrm{ch}}_i \cdot \eta^{\mathrm{dis}}_i$ is the round-trip efficiency, typically 0.80–0.90 for lithium-ion systems. Because $E_{i,t}$ depends on all prior decisions, this constraint is what makes the problem a **multi-period MILP** rather than a separable set of independent single-step problems.

**Battery SOC bounds:**

$$0 \le E_{i,t} \le E^{\mathrm{bat}}_{\max,i} \quad \forall t \in \mathcal{T},\ i \in \mathcal{P}$$

The lower bound prevents over-discharge, which would damage battery cells; a stricter lower bound (e.g. $E_{\min} = 0.2 \cdot E^{\mathrm{bat}}_{\max}$) is often imposed in practice to preserve cycle life. The upper bound is the usable energy capacity of the installed BESS. Together, these constraints define the **feasible operating window** for the battery and limit how aggressively the optimizer can exploit storage.

**Battery power bounds:**

$$x^{\mathrm{ch}}_{i,t} \le P^{\mathrm{ch}}_{\max,i}, \qquad x^{\mathrm{dis}}_{i,t} \le P^{\mathrm{dis}}_{\max,i} \quad \forall t \in \mathcal{T},\ i \in \mathcal{P}$$

These rate constraints reflect the physical limits of the battery inverter and the C-rate of the cells. They prevent the optimizer from instantaneously charging or discharging the full capacity in a single 15-minute step — which would not be physically realisable — and ensure that the scheduled power ramps are within the safe operating envelope of the hardware.

**Non-simultaneous charging/discharging** (optional, for operational realism):

$$x^{\mathrm{ch}}_{i,t} \cdot x^{\mathrm{dis}}_{i,t} = 0 \quad \forall t \in \mathcal{T},\ i \in \mathcal{P}$$

> Although this constraint introduces non-convexity (it can be linearised with binary variables and big-M reformulation, converting the LP to a MILP), it is commonly included to reflect the fact that most battery inverters cannot simultaneously charge and discharge. Without it, the optimizer may exploit the efficiency asymmetry between $\eta^{\mathrm{ch}}$ and $\eta^{\mathrm{dis}}$ by cycling the battery at zero net SOC change — a physically meaningless but mathematically feasible artefact.

---

### 4.3.3 Outcome

Solving the Stage 1 problem yields the **individual minimum cost** $C^{\mathrm{ind}*}_i$ for each participant:

$$C^{\mathrm{ind}*}_i = \sum_{t \in \mathcal{T}} \Delta t \left( p^{\mathrm{imp}}_{i,t}\, P^{\mathrm{imp}*}_{i,t} - p^{\mathrm{exp}}_{i,t}\, P^{\mathrm{exp}*}_{i,t} \right)$$

The asterisk on all quantities denotes optimal values returned by the solver. $C^{\mathrm{ind}*}_i$ is the **lowest energy bill participant $i$ can achieve acting alone**, given their local resources and tariff structure. For prosumers, the optimal battery schedule shifts self-consumption to high-price periods and front-loads charging during low-price or high-generation windows. For consumers, $C^{\mathrm{ind}*}_i$ reduces to the fixed cost $\sum_t \Delta t\, p^{\mathrm{imp}}_{i,t} L_{i,t}$ (no dispatch freedom).

These individual costs serve a dual role in Stage 2: they act as **cost caps** in the Pareto fairness constraint ($C^{\mathrm{rec}}_i \leq C^{\mathrm{ind}*}_i$), and as a **counterfactual benchmark** for evaluating the monetary benefit of REC participation for each member.


---

## 4.4 Stage 2: Collective REC Optimization (Minimize Community Costs with Fairness Guarantees)

### 4.4.1 Rationale and Problem Formulation

In Stage 2, the REC as a collective entity optimizes the dispatch of all distributed resources (batteries, flexible loads, and internal electricity trades) to minimize **aggregate community cost**. Critically, the formulation enforces a Pareto-optimal fairness constraint: each participant's new bill must not exceed their Stage 1 individual cost $^{[3]}$.

This approach is grounded in EU directives mandating that energy communities deliver benefits to all members $^{[4]}$. The internal pricing mechanism — the rate at which prosumers sell to other community members — is set as a uniform, time-varying internal price $p^{\mathrm{p2p}}_t$ applying to all intra-community trades.

**Additional parameters (beyond Stage 1):**
- $C^{\mathrm{ind}*}_i$: individual minimum costs from Stage 1
- $p^{\mathrm{p2p}}_t$: internal REC electricity price (€/kWh) for peer-to-peer transfers
- $p^{\mathrm{grid}}_t$: local grid access tariff (€/kWh) for self-consumed energy; in Austria, typically 30–60% of the standard grid tariff $^{[5]}$

**Additional decision variables:**
- $E^{\mathrm{sale}}_{i,j,t} \geq 0$: energy (kWh) exported by prosumer $i$ to participant $j$ at time $t$
- $E^{\mathrm{purch}}_{i,j,t} \geq 0$: energy (kWh) purchased by participant $i$ from prosumer $j$ at time $t$

**Objective function (community cost minimization):**

$$\min \sum_{i \in \mathcal{C} \cup \mathcal{P}} \sum_{t \in \mathcal{T}} \Delta t \left[ p^{\mathrm{imp}}_{i,t}\, P^{\mathrm{imp}}_{i,t} - p^{\mathrm{exp}}_{i,t}\, P^{\mathrm{exp}}_{i,t} + p^{\mathrm{grid}}_t\, E^{\mathrm{self}}_{i,t} \right]$$

Unlike Stage 1, the $\min$ here operates over **all participants simultaneously** — the outer sum $\sum_{i \in \mathcal{C} \cup \mathcal{P}}$ aggregates costs across the entire community, so the optimizer can trade off one participant's import against another's export. The three terms inside the brackets decompose the community's total financial exposure at each time step $t$:

- $p^{\mathrm{imp}}_{i,t} P^{\mathrm{imp}}_{i,t}$: **grid import cost** for participant $i$ — reduced when internal trades or battery discharge can substitute for external supply.
- $-\, p^{\mathrm{exp}}_{i,t} P^{\mathrm{exp}}_{i,t}$: **grid export revenue** — prosumers earn a feed-in premium for surplus generation not consumed internally; the negative sign means this reduces total community cost.
- $p^{\mathrm{grid}}_t\, E^{\mathrm{self}}_{i,t}$: **local grid access charge** on internally shared energy — in Austria, energy traded within a REC still incurs a reduced grid usage fee (typically 30–60% of the full tariff) $^{[5]}$. This term penalises internal sharing slightly, but at a rate far below the full import tariff, making internal trade economically attractive.

The key difference from Stage 1 is that $E^{\mathrm{self}}_{i,t}$ — the internally sourced share of participant $i$'s consumption — is now a decision variable influenced by battery dispatch and peer-to-peer trade allocations. The optimizer therefore jointly schedules all batteries and all internal transfers to maximize community-wide self-consumption (replacing expensive grid imports with cheaper internal renewable supply) while respecting the individual fairness constraints from Stage 1.


### 4.4.2 Constraints

**Modified energy balance — prosumers** $^{[7]}$:

$$P^{\mathrm{imp}}_{i,t} - P^{\mathrm{exp}}_{i,t} + x^{\mathrm{dis}}_{i,t} - x^{\mathrm{ch}}_{i,t} = L_{i,t} - G_{i,t} - \sum_{j \neq i} E^{\mathrm{sale}}_{i,j,t} + \sum_{j \neq i} E^{\mathrm{purch}}_{i,j,t}$$

A prosumer with surplus generation can either export to the grid or sell internally; the optimizer chooses based on price signals and battery availability.

**Modified energy balance — consumers** $^{[7]}$:

$$P^{\mathrm{imp}}_{i,t} - P^{\mathrm{exp}}_{i,t} = L_{i,t} - \sum_{j \in \mathcal{P}} E^{\mathrm{purch}}_{i,j,t}$$

Consumers can only reduce grid import by purchasing from prosumers.

**Internal trade balance (bilateral symmetry)** $^{[7]}$:

$$\sum_{j \neq i} E^{\mathrm{sale}}_{i,j,t} = \sum_{j \neq i} E^{\mathrm{purch}}_{j,i,t} \quad \forall t \in \mathcal{T}$$

Total energy sold by participant $i$ must equal total energy purchased from $i$ by all other members — ensuring energy conservation within the community.

**Self-consumption accounting** $^{[6]}$:

$$E^{\mathrm{self}}_{i,t} = \min \left( L_{i,t},\ G_{i,t} + \sum_{j \neq i} E^{\mathrm{purch}}_{i,j,t} \right)$$

**Fairness constraint (Pareto optimality guarantee)** $^{[8]}$:

Each participant's Stage 2 bill must not exceed their Stage 1 individual cost:

$$C^{\mathrm{rec}}_i = \sum_{t \in \mathcal{T}} \Delta t \left[ p^{\mathrm{imp}}_{i,t} P^{\mathrm{imp}}_{i,t} - p^{\mathrm{exp}}_{i,t} P^{\mathrm{exp}}_{i,t} + p^{\mathrm{p2p}}_t \left( \sum_j E^{\mathrm{purch}}_{i,j,t} - \sum_j E^{\mathrm{sale}}_{i,j,t} \right) \right] \leq C^{\mathrm{ind}}_i \quad \forall i$$

**Battery dynamics and bounds** (identical to Stage 1):

$$E_{i,t} = E_{i,t-1} + \eta^{\mathrm{ch}}_i\, x^{\mathrm{ch}}_{i,t}\, \Delta t - \frac{1}{\eta^{\mathrm{dis}}_i}\, x^{\mathrm{dis}}_{i,t}\, \Delta t, \qquad 0 \le E_{i,t} \le E^{\mathrm{bat}}_{\max,i}$$

---

### 4.4.3 Outcome

Solving the Stage 2 problem yields:
- Optimized battery schedules $x^{\mathrm{ch}*}_{i,t}$, $x^{\mathrm{dis}*}_{i,t}$ for all prosumers
- Optimized internal energy trades $E^{\mathrm{sale}*}_{i,j,t}$, $E^{\mathrm{purch}*}_{i,j,t}$
- Collective community cost $C^{\mathrm{REC}} = \sum_i C^{\mathrm{rec}}_i$

By construction, $C^{\mathrm{REC}} \leq \sum_i C^{\mathrm{ind}}_i$ (community as a whole benefits), and the fairness constraint guarantees:

$$C^{\mathrm{rec}}_i \leq C^{\mathrm{ind}}_i \quad \forall i \in \mathcal{C} \cup \mathcal{P}$$

---

## 4.5 Model Predictive Control (MPC) Implementation

Uncertainty in PV production and electricity demand is addressed using a **receding-horizon MPC approach** $^{[2]}$, essential for managing the volatile nature of renewable energy sources and unpredictable consumption patterns.

The MPC framework operates iteratively:

1. **Forecasting**: A 24-hour horizon (96 time steps at 15-minute resolution) is defined using day-ahead forecasts for PV generation, load, and battery availability $^{[2]}$. Demand forecasts are taken from measurements of the previous week; PV production profiles from real historic weather forecasts $^{[9]}$.
2. **Optimization**: The MILP computes optimal battery schedules and internal energy flows over the full 24-hour horizon $^{[2]}$.
3. **Execution**: Only the decision for the **first 15-minute time step** is implemented in reality $^{[2]}$.
4. **Rolling Update**: The horizon advances by one 15-minute interval, forecasts are updated, and the MILP is re-solved for the new window $^{[10]}$.

This iterative procedure allows continuous adaptation to forecast errors while preserving computational efficiency and managing system dynamics $^{[11]}$.

---

## 4.6 Computational Performance

The MILP formulation, embedded within the MPC framework, is computationally compact:

| Metric | Value |
|--------|-------|
| Simulation duration | 1 week |
| MILP re-solves per day | 96 (one per 15-minute interval) |
| Total computation time | ~696 seconds |
| Hardware | Intel® Core™ i7-1065G7 @ 1.30 GHz, 16 GB RAM |
| Solver | Gurobi 9.5 $^{[12]}$ |

This efficiency enables systematic scenario analysis across multiple REC configurations within reasonable timeframes, particularly for real-time operational processes $^{[10]}$.

---

## 4.7 Discussion: The Two-Stage Approach and Fairness Guarantees

### 4.7.1 Why Two Stages?

The two-stage hierarchy addresses a fundamental tension in energy community design:

- **Individual autonomy**: Each participant has its own electricity tariff, retail supply contract, and metering infrastructure. Stage 1 respects this autonomy by computing each participant's standalone optimum.
- **Collective efficiency**: A centrally coordinated REC can achieve economies of scale through battery pooling, synchronised demand-side flexibility, and internal energy sharing. Stage 2 captures these collective gains.

By separating these concerns, the framework ensures that participants are not coerced into accepting worse individual outcomes in exchange for collective benefits. This transparency is essential for building trust and participation rates in real-world RECs $^{[13]}$.

---

### 4.7.2 Pareto Optimality and Fairness

The fairness constraint $C^{\mathrm{rec}}_i \leq C^{\mathrm{ind}}_i$ is a **Pareto optimality condition**: no individual can be made better off without making another worse off. In the context of RECs $^{[3,4]}$:

- If a participant's cost **would increase** due to collective optimization, that strategy is forbidden.
- If a participant's cost **decreases**, they gain; if it remains constant, they face no penalty for participation.
- The aggregate cost must decrease because of the renewable generation and storage assets already present in the community.

This approach aligns with **EU Directive 2019/944** $^{[14]}$, which mandates that energy communities deliver benefits to members and do not prioritise profit extraction $^{[15]}$. In practice, savings from collective operation are often small (~5–15% for early-stage RECs without substantial storage) but are **guaranteed positive at the aggregate level**.

---

### 4.7.3 Internal Pricing and Dynamic Allocation

The internal REC price $p^{\mathrm{p2p}}_t$ is a design choice with significant implications $^{[16]}$:

| Pricing Strategy | Description | Trade-off |
|-----------------|-------------|-----------|
| **Fixed mid-market rate** (e.g., 0.10 €/kWh) | Simple, transparent $^{[17]}$ | May not reflect true supply–demand dynamics |
| **Dynamic rate** (supply–demand ratio) | More efficient $^{[18]}$ | Requires real-time communication; perceived as less fair |

For this case study, a **fixed rate** is assumed (typically the average of the grid import and export prices), providing stability and simplicity. Sensitivity analysis (Section 6) explores the impact of different pricing strategies on community surplus and member satisfaction.

---

## 4.8 Input Data Sources and Time Series

The optimization requires the following inputs:

| Input | Description | Source |
|-------|-------------|--------|
| Load profiles $L_{i,t}$ | 15-minute resolved electricity demand per participant | Smart meter data or standard load profiles (SLPs) $^{[19]}$ |
| PV generation profiles $G_{i,t}$ | 15-minute resolved solar irradiance converted to power output | PVGIS or SimBench datasets $^{[9,20]}$ |
| Import tariff $p^{\mathrm{imp}}_{i,t}$ | Austrian wholesale prices plus retail markups and taxes | EXAA or EPEX Spot $^{[21,22]}$ |
| Export price $p^{\mathrm{exp}}_{i,t}$ | Feed-in compensation, varies by contract | EXAA or EPEX Spot $^{[21,22]}$ |
| Battery parameters | Capacity (kWh), power limits (kW), efficiency, cost (€/kWh) | Manufacturer data $^{[23,24]}$ |

**Typical modern lithium-ion battery parameters** $^{[24]}$:
- Capacity: $E_{\max} = 5$–$10$ kWh
- Power limit: $P_{\max} = 2$–$3$ kW
- Efficiency: $\eta_{\mathrm{ch}} / \eta_{\mathrm{dis}} \approx 0.90$–$0.95$

> Market prices can be volatile due to renewable energy integration $^{[21,22]}$. These inputs are discussed in detail in Section 5 (Case Study Data) and integrated into the solver framework described in Section 6 (Computational Experiments).

---

## References

| # | Citation |
|---|----------|
| [1,2,9,10,11,12] | Frieß et al., "Optimization and Simulation for the Daily Operation of Renewable Energy Communities," 2024. |
| [3,4,6,7,8,13,16,17,18] | Rocha et al., "A Three-Stage Model to Manage Energy Communities, Share Benefits and Provide Local Grid Services," *Energies*, 2023. |
| [5] | Transposition of European Guidelines for Energy Communities into Austrian Law: A Comparison and Discussion of Issues and Positive Aspects, 2021. |
| [14] | DIRECTIVE (EU) 2019/944 of the European Parliament and of the Council of 5 June 2019 on common rules for the internal market for electricity (recast), 2019. |
| [15] | Ahmed et al., "Renewable Energy Communities: Towards a new sustainable model of energy production and sharing," *Energy Strategy Reviews*, 2023. |
| [19,20] | Spalthoff et al., "SimBench: Open source time series of power load, storage and generation for the simulation of electrical distribution grids," 2017. |
| [21,22] | O'Connor et al., "A Review of Electricity Price Forecasting Models in the Day-Ahead, Intra-Day, and Balancing Markets," *Energies*, 2025. |
| [23] | Verifying the Targets, 2020. |
| [24] | Alnaser et al., "Residential community with PV and batteries: Reserve provision under grid constraints," *International Journal of Electrical Power and Energy Systems*, 2020. |